# Pixel cohort: the 150 prod150 words, optimized directly in pixels

This notebook makes one image per word by optimizing the **pixels** directly against a guidance model, with no strokes and no generator. It uses the recipe from the CLIPAG authors' own text-to-image notebook (`CLIPAG_T2I.ipynb`): 100 AdamW steps on a 224×224 image, scored each step on 32 augmented whole-image views and 32 random crops. `clipag_cohort/pixels.py` lists every setting.

Two choices keep it paired with the stroke runs: the prompt is `A drawing of {word}.` as in prod150, and every word starts from the same faint noise image (seed 0).

Together with the stroke runs this fills a 2×2 design:

| | Strokes (CLIPDraw) | Pixels (this notebook) |
|---|---|---|
| **CLIP** | `prod150_v1` | `GUIDANCE_MODEL="clip"`, `RUN_ID="clip150px_v1"` |
| **CLIPAG** | `clipag150_v1` | `GUIDANCE_MODEL="clipag"`, `RUN_ID="clipag150px_v1"` |

Standard CLIP is expected to produce noise-like images that score well with CLIP but look like nothing to people. That makes it a test of whether the judge is fooled by the same tricks.

**Order:** smoke test first (`RUN_MODE="smoke_test"`, `WORD_LIMIT=2`, throwaway `RUN_ID`), then the full runs. No DiffVG compile is needed, so setup is quick. Results go to `runs/<RUN_ID>/images/part<P>/<NNN>_<word>/` in the same layout as the stroke runs, so the judges and the synthesis notebook read them unchanged. If Colab disconnects, rerun all cells with the same settings: finished words are skipped.

In [ ]:
# @title 1. Settings
RUN_ID = "clipag150px_v1"  # @param {type:"string"}
GUIDANCE_MODEL = "clipag"  # @param ["clipag", "clip"]
RUN_MODE = "full"  # @param ["full", "smoke_test"]
SUBSET = "all"  # @param ["all", "control15"]
WORD_LIMIT = 0  # @param {type:"integer"}
PERSON = 1  # @param {type:"integer"}
NUM_PEOPLE = 1  # @param {type:"integer"}
PROJECT_FOLDER = "MyDrive/ExperimentResults"  # @param {type:"string"}
# "drive" keeps the 1.3 GB CLIPAG checkpoint on Drive (downloaded once).
# "colab" keeps it on Colab's temporary disk: no Drive space, but it re-downloads every session.
CHECKPOINT_LOCATION = "drive"  # @param ["drive", "colab"]
SOURCE_REF = "experiment/clipdraw-clipag"  # @param {type:"string"}
REPO_URL = "https://github.com/ArmitaAfr/Visibility_Thresholds.git"

if WORD_LIMIT and RUN_MODE != "smoke_test":
    raise ValueError("WORD_LIMIT is only for smoke tests. Set it to 0 for a full run.")

## 2. Drive, source and word list

In [ ]:
import os, re, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = (Path("/content/drive") / PROJECT_FOLDER).resolve()
if not DRIVE_ROOT.is_dir():
    raise ValueError(f"Project folder not found on Drive: {DRIVE_ROOT}")

SRC = Path("/content/Visibility_Thresholds")
if not SRC.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(SRC)], check=True)
subprocess.run(["git", "fetch", "origin", SOURCE_REF], cwd=SRC, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=SRC, check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SRC, text=True).strip()
print("Source commit:", SOURCE_COMMIT)

sys.path.insert(0, str(SRC))
import importlib
import clipag_cohort.cohort as cohort
import clipag_cohort.pixels as pixels
importlib.reload(cohort); importlib.reload(pixels)

subprocess.run(["git", "fetch", "origin", cohort.SAMPLED_WORDS_COMMIT], cwd=SRC, check=True)
words_bytes = subprocess.check_output(
    ["git", "show", f"{cohort.SAMPLED_WORDS_COMMIT}:{cohort.SAMPLED_WORDS_PATH}"], cwd=SRC)
ALL_WORDS = cohort.load_words(words_bytes)

settings = cohort.pixel_settings(run_id=RUN_ID, guidance_model=GUIDANCE_MODEL, run_mode=RUN_MODE,
                                 subset=SUBSET, people=NUM_PEOPLE, recipe=pixels.RECIPE)
settings["word_limit"] = WORD_LIMIT
RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID
if RUN_ID == cohort.PROD150["run_id"] or (RUN_DIR / "stages").exists() and not (RUN_DIR / "cohort_settings.json").exists():
    raise ValueError(f"{RUN_DIR} belongs to another pipeline. Choose a new RUN_ID.")
cohort.check_or_write_json(RUN_DIR / "cohort_settings.json", settings)

words = cohort.assignment(cohort.select_words(ALL_WORDS, SUBSET), PERSON, NUM_PEOPLE)
if WORD_LIMIT:
    words = words[:WORD_LIMIT]
PART_DIR = RUN_DIR / "images" / f"part{PERSON}"
STAGING_DIR = RUN_DIR / "staging" / f"part{PERSON}"
PLAN = cohort.plan_part(words, PART_DIR, GUIDANCE_MODEL, substrate="pixels")
todo = [p for p in PLAN if not p[3]]
print(f"Run {RUN_ID} | {GUIDANCE_MODEL} pixels | part {PERSON}/{NUM_PEOPLE} | {len(PLAN)} words, "
      f"{len(PLAN) - len(todo)} done, {len(todo)} to make")

## 3. Load the guidance model

Installs OpenAI CLIP at the commit `clipdraw_clipag.ipynb` pins, then runs that notebook's model-loading cell, so CLIPAG is downloaded, checked (size and MD5) and loaded strictly exactly as in the stroke runs. DiffVG is not needed here.

In [ ]:
import json, platform, importlib.metadata
import torch, torchvision

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime: Runtime → Change runtime type → GPU.")
device = torch.device("cuda")

source_nb = json.loads((SRC / "clipdraw_clipag.ipynb").read_text(encoding="utf-8"))
code_cells = ["".join(c["source"]) for c in source_nb["cells"] if c["cell_type"] == "code"]

def cell_with(marker):
    found = [c for c in code_cells if marker in c]
    if len(found) != 1:
        raise RuntimeError(f"Expected one clipdraw_clipag.ipynb cell containing {marker!r}, found {len(found)}.")
    return found[0]

CLIP_COMMIT = re.search(r'CLIP_COMMIT = "([0-9a-f]{40})"', cell_with("DIFFVG_COMMIT = ")).group(1)

def run(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("+", " ".join(command))
    return subprocess.run(command, cwd=cwd, env=env, check=True)

CLIP_DIR = Path("/content/CLIP")
if not CLIP_DIR.exists():
    run(["git", "clone", "--filter=blob:none", "https://github.com/openai/CLIP.git", CLIP_DIR])
run(["git", "fetch", "origin", CLIP_COMMIT, "--depth", "1"], cwd=CLIP_DIR)
run(["git", "checkout", "--detach", CLIP_COMMIT], cwd=CLIP_DIR)
run([sys.executable, "-m", "pip", "install", "ftfy", "regex"])
run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", CLIP_DIR])
if str(CLIP_DIR) not in sys.path:
    sys.path.insert(0, str(CLIP_DIR))
import clip

environment = {"python": platform.python_version(), "pytorch": torch.__version__,
               "torchvision": torchvision.__version__, "pytorch_cuda": torch.version.cuda,
               "gpu": torch.cuda.get_device_name(0), "clip_commit": CLIP_COMMIT}

# Names the model-loading cell reads.
EXPERIMENT_MODE = "cohort"
NORMALIZE_CLIP_INPUTS = False  # only affects that cell's stroke augmentations, unused here
if CHECKPOINT_LOCATION == "drive":
    CLIPAG_CHECKPOINT_PATH = str(DRIVE_ROOT / "checkpoints" / "CLIPAG_ViTB32.pt")
else:
    CLIPAG_CHECKPOINT_PATH = "/content/checkpoints/CLIPAG_ViTB32.pt"
DOWNLOAD_CLIPAG_CHECKPOINT = True
exec(compile(cell_with("def load_guidance_model"), "clipdraw_clipag.ipynb[load_guidance_model]", "exec"), globals())

load_guidance_model()
print(json.dumps(guidance_metadata, indent=2))

## 4. Optimize the assigned words

Each result is written to a staging folder and moved into `images/` only once complete. The preview shows the current image every 20 steps.

In [ ]:
import csv, gc, hashlib, shutil, time
import numpy as np
from PIL import Image
from IPython.display import display, clear_output

effective = dict(pixels.SMOKE) if RUN_MODE == "smoke_test" else {
    k: pixels.RECIPE[k] for k in ("steps", "num_views", "num_patches")}
SEED = cohort.PROD150["seed"]

def to_pil(image):
    return Image.fromarray((image[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255).round().astype(np.uint8))

init = pixels.initial_image(SEED)
cohort.check_or_write_json(RUN_DIR / "initial_image.json", {
    "sha256": hashlib.sha256(init.numpy().tobytes()).hexdigest(), "seed": SEED,
    "definition": pixels.RECIPE["init"]})

if STAGING_DIR.exists():
    shutil.rmtree(STAGING_DIR)
STAGING_DIR.mkdir(parents=True)
LOG = RUN_DIR / "logs" / f"part{PERSON}.csv"
LOG.parent.mkdir(parents=True, exist_ok=True)
log_fields = ["utc", "folder", "word", "status", "optimization_seconds", "final_loss", "error"]
if not LOG.exists():
    with LOG.open("w", newline="") as f:
        csv.DictWriter(f, fieldnames=log_fields).writeheader()

def log(row):
    with LOG.open("a", newline="") as f:
        csv.DictWriter(f, fieldnames=log_fields).writerow(
            {"utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), **row})

failures = []
for n, (index, w, folder, _) in enumerate(todo, 1):
    word = w["word"]
    progress = f"{RUN_ID} | part {PERSON} | {n}/{len(todo)} | {word} ({w['group']})"
    print(progress)
    staging = STAGING_DIR / folder.name
    try:
        prompt = cohort.PROD150["prompt_template"].format(word)
        with torch.no_grad():
            text_features = model.encode_text(tokenize([prompt]).to(device))

        def preview(step, image, losses):
            if step % 20 == 0 or step == effective["steps"] - 1:
                clear_output(wait=True)
                print(progress)
                display(to_pil(image).resize((224, 224)))
                print(f"Step {step + 1}/{effective['steps']} | global {losses['global']:.4f} | patch {losses['patch']:.4f}")

        started = time.perf_counter()
        best, best_step, history = pixels.optimize_pixels(
            model.encode_image, text_features, seed=SEED, device=device, on_step=preview, **effective)
        elapsed = time.perf_counter() - started
        if not all(np.isfinite(h["total"]) for h in history):
            raise RuntimeError("Nonfinite loss.")

        staging.mkdir(parents=True)
        # Same file name as the stroke runs, so the judge pipeline finds it.
        to_pil(best).save(staging / "clipdraw_final.png")
        (staging / "loss_history.json").write_text(json.dumps(history))
        record = cohort.pixel_record(
            word_row=w, index=index, settings=settings, person=PERSON, source_commit=SOURCE_COMMIT,
            prompt=prompt, seed=SEED, effective=effective, guidance=json.loads(json.dumps(guidance_metadata)),
            environment=environment, elapsed_seconds=elapsed, best_step=best_step,
            best_global_loss=history[best_step]["global"], steps_run=len(history))
        (staging / "experiment.json").write_text(json.dumps(record, indent=2))
        cohort.publish(staging, folder)
        log({"folder": folder.name, "word": word, "status": "ok", "optimization_seconds": round(elapsed, 1),
             "final_loss": round(history[best_step]["global"], 6), "error": ""})
    except Exception as exc:
        failures.append((word, f"{type(exc).__name__}: {exc}"))
        log({"folder": folder.name, "word": word, "status": "error", "optimization_seconds": "",
             "final_loss": "", "error": failures[-1][1]})
        print("FAILED:", failures[-1])
        shutil.rmtree(staging, ignore_errors=True)
    finally:
        gc.collect()
        torch.cuda.empty_cache()

PLAN = cohort.plan_part(words, PART_DIR, GUIDANCE_MODEL, substrate="pixels")
remaining = [w["word"] for _, w, _, done in PLAN if not done]
if remaining:
    print(f"{len(remaining)} words not finished: {remaining}. Rerun this notebook to retry them.")
else:
    receipt = {"run_id": RUN_ID, "person": PERSON, "folders": [f.name for _, _, f, _ in PLAN],
               "png_sha256": {f.name: hashlib.sha256((f / "clipdraw_final.png").read_bytes()).hexdigest()
                              for _, _, f, _ in PLAN}}
    cohort.check_or_write_json(RUN_DIR / "generation" / f"part{PERSON}.json", receipt)
    print(f"Part {PERSON} complete: {len(PLAN)} images in {PART_DIR}")

## 5. Next

Judge these images like the stroke runs (`EXISTING_ROOTS` = this run's `images` folder), or let `clipag_synthesis.ipynb` collect every run into one folder first and judge that once.